In [ ]:
import tarfile
import pandas as pd


def process_df(df: pd.DataFrame) -> pd.DataFrame:
    # Doc: https://s3.opensky-network.org/data-samples/states/README.txt
    df = df.copy()
    df.query("~`onground`", inplace=True)
    df.drop(
        columns=["alert", "spi", "squawk", "lastposupdate", "lastcontact", "onground"],
        inplace=True,
    )
    df["time"] = pd.to_datetime(df["time"], unit="s")
    df.dropna(how="any", inplace=True)
    df.set_index(["icao24", "time"], inplace=True)
    df.sort_index(inplace=True)
    df.rename(columns={"geoaltitude": "alt"}, inplace=True)

    df = df[["callsign", "lat", "lon", "alt"]]

    return df

In [ ]:
data_dir = "/home/user/Downloads/opensky_data"
out_path = "/home/user/Downloads/data_opensky_2017-06-15.csv"

lat_min = 45.351944
lon_min = 5.449722
lat_max = 48.189320
lon_max = 10.890690

In [ ]:
import os

import pandas as pd
from tqdm import tqdm


dfs = []
for filename in tqdm(os.listdir(data_dir)):
    path = f"{data_dir}/{filename}"
    with tarfile.open(path, "r") as tar:
        file = tar.extractfile(filename.replace("tar", "gz"))
        df = pd.read_csv(file, compression="gzip")
        df = process_df(df)
        df = df.query(
            "(@lat_min <= `lat` <= @lat_max) and (@lon_min <= `lon` <= @lon_max)"
        )
        dfs.append(df)
df = pd.concat(dfs).sort_index()
df.to_csv(out_path)